# 29 — Dense RGB review evidence for the six-creator external benchmark

**Runtime: CPU.** Run all once. T4 is unnecessary: this notebook downloads 360p sources and makes contact sheets, without model training or pose extraction. It outputs one ZIP automatically. The six sources remain external-test only.

The 37 intervals are **draft locators**. The ZIP contains RGB frames sampled at 2 FPS around each locator, with timestamps and an index, so visible movement, rests, talking, and boundaries can be reviewed. It does **not** lock labels, score models, or include raw videos. Downloaded videos stay in the temporary Colab runtime. No manual upload or review widget is needed.

In [ ]:
%pip install -q --upgrade "yt-dlp[default]" opencv-python-headless pandas==2.2.3 tqdm

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, re, shutil, subprocess, urllib.request, zipfile
import cv2
import numpy as np
import pandas as pd
from google.colab import files
from IPython.display import display
from tqdm.auto import tqdm

PINNED_DATA_COMMIT = '752e741c96a1647b880e6222aa4989a03e0b18bb'
RAW_BASE = f'https://raw.githubusercontent.com/bariscelikk1/cALorie/{PINNED_DATA_COMMIT}'
EXPECTED_SHA256 = {
    'candidate_intervals_v1.csv': 'e82424985487effb9d06a8339c9be15e0008702869ae1565c3698637fcdc0381',
    'candidates.csv': 'aa3063810fa3b0e9fe169c04281ef65cc33a442d18be0fa0e5fec7669e06268b',
}
WORK_ROOT = Path('/content/calorie_creator_benchmark_dense_review_work')
VIDEO_ROOT = WORK_ROOT / 'videos'
FRAME_ROOT = WORK_ROOT / 'frames'
OUTPUT_ROOT = Path('/content/calorie_creator_benchmark_dense_review_evidence')
CONTACT_ROOT = OUTPUT_ROOT / 'dense_contact_sheets'
for directory in (WORK_ROOT, VIDEO_ROOT, FRAME_ROOT, OUTPUT_ROOT, CONTACT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
def ensure_deno():
    machine = platform.machine().lower()
    if machine in {'x86_64', 'amd64'}:
        asset = 'deno-x86_64-unknown-linux-gnu.zip'
    elif machine in {'aarch64', 'arm64'}:
        asset = 'deno-aarch64-unknown-linux-gnu.zip'
    else:
        raise RuntimeError(f'Unsupported Colab architecture for Deno: {machine}')
    deno_root = WORK_ROOT / 'deno_runtime'
    deno_path = deno_root / 'deno'
    if not deno_path.exists():
        deno_root.mkdir(parents=True, exist_ok=True)
        deno_zip = deno_root / 'deno.zip'
        urllib.request.urlretrieve(
            f'https://github.com/denoland/deno/releases/latest/download/{asset}', deno_zip
        )
        with zipfile.ZipFile(deno_zip) as archive:
            archive.extract('deno', deno_root)
        deno_path.chmod(0o755)
    result = subprocess.run(
        [str(deno_path), '--version'], text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False,
    )
    if result.returncode:
        raise RuntimeError('Deno runtime failed:\n' + result.stdout)
    print('JavaScript runtime:', result.stdout.splitlines()[0])
    return deno_path


DENO_BIN = ensure_deno()
YTDLP_RUNTIME_ARGS = [
    '--js-runtimes', f'deno:{DENO_BIN}',
    '--remote-components', 'ejs:github',
]

print('Runtime: CPU; pinned data commit:', PINNED_DATA_COMMIT)

## Fetch the pinned draft locators

Both CSV inputs come from a fixed GitHub commit and must match their SHA-256 hashes. Rejected sources are excluded. The notebook refuses to run if the six expected source IDs or the 37 draft intervals change.

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def pinned_download(name):
    destination = WORK_ROOT / name
    if not destination.exists() or sha256(destination) != EXPECTED_SHA256[name]:
        temporary = destination.with_suffix(destination.suffix + '.part')
        temporary.unlink(missing_ok=True)
        urllib.request.urlretrieve(f'{RAW_BASE}/ml/data/youtube_creator_benchmark_v1/{name}', temporary)
        assert sha256(temporary) == EXPECTED_SHA256[name], name
        temporary.replace(destination)
    return destination

input_paths = {name: pinned_download(name) for name in EXPECTED_SHA256}
intervals = pd.read_csv(input_paths['candidate_intervals_v1.csv'])
all_sources = pd.read_csv(input_paths['candidates.csv'])
expected_ids = {'ojs-UW8ULhs','fJQmoPKvjTg','Kw89u6WtPC8','phFvFuLUFEw','lKHGBuanBsY','7-mVPCVJMtI'}
assert len(intervals) == 37 and intervals.interval_id.is_unique
assert set(intervals.video_id) == expected_ids
assert (intervals.review_status == 'draft').all()
assert (intervals.needs_second_review.astype(str).str.lower() == 'true').all()
assert (intervals.benchmark_ready.astype(str).str.lower() == 'false').all()
sources = all_sources.loc[all_sources.video_id.isin(expected_ids)].copy()
assert len(sources) == 6 and sources.video_id.is_unique
assert (sources.benchmark_role == 'locked_external_test').all()
for col in ('training_allowed','calibration_allowed','threshold_selection_allowed'):
    assert not sources[col].astype(str).str.lower().isin(['true','1','yes']).any()
assert (intervals.end_s > intervals.start_s).all()
print('Draft locators:', len(intervals), '| reserved external-test videos:', len(sources))
display(intervals.groupby('canonical_class').interval_id.count().rename('candidate_regions'))

## Download temporary 360p videos

This uses the same Deno/EJS and yt-dlp fallback sequence as Notebook 28B. Completed downloads are reused if a Colab cell is rerun. A video is accepted only if ffprobe decodes its stream and its height is 360. YouTube can still deny a source at service level; the last errors are reported clearly in that case.

In [ ]:
DOWNLOAD_ATTEMPTS = [
    {
        'name': 'proven H.264 360p format 134',
        'format': '134',
        'extra': [],
    },
    {
        'name': '360p combined MP4',
        'format': '18/best[height<=360][ext=mp4]/best[height<=360]',
        'extra': [],
    },
    {
        'name': 'Notebook 23 TV-client fallback',
        'format': '18/best[height<=360]/worst',
        'extra': ['--extractor-args', 'youtube:player_client=tv'],
    },
    {
        'name': 'Notebook 23 embedded-client fallback',
        'format': '18/best[height<=360]/worst',
        'extra': ['--extractor-args', 'youtube:player_client=web_embedded'],
    },
]


VIDEO_SUFFIXES = {'.mp4','.mkv','.webm','.mov'}

def probe_video(path):
    result = subprocess.run([
        'ffprobe','-v','error','-select_streams','v:0',
        '-show_entries','stream=width,height,nb_frames:format=duration',
        '-of','json',str(path),
    ], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)
    if result.returncode:
        raise RuntimeError(f'ffprobe failed for {path.name}: {result.stdout[-1200:]}')
    data = json.loads(result.stdout)
    stream = data['streams'][0]
    return {'width':int(stream['width']), 'height':int(stream['height']),
            'duration_s':float(data['format'].get('duration') or 0)}

def valid_video(path):
    try:
        info = probe_video(path)
        return info['height'] == 360 and info['duration_s'] >= 120
    except Exception:
        return False

def download_360p(row):
    existing = sorted(p for p in VIDEO_ROOT.glob(f'{row.video_id}.*')
                      if p.suffix.lower() in VIDEO_SUFFIXES and p.stat().st_size > 1_000_000)
    if len(existing) == 1 and valid_video(existing[0]):
        print('Reusing:', existing[0].name)
        return existing[0]
    for path in existing:
        path.unlink(missing_ok=True)
    failures = []
    output_template = VIDEO_ROOT / f'{row.video_id}.%(ext)s'
    for attempt in DOWNLOAD_ATTEMPTS:
        for partial in VIDEO_ROOT.glob(f'{row.video_id}*.part*'):
            partial.unlink(missing_ok=True)
        command = [
            'yt-dlp','--no-playlist','--no-progress','--retries','10',
            '--fragment-retries','10','--retry-sleep','fragment:exp=1:8',
            '--socket-timeout','30','-f',attempt['format'],
            '--merge-output-format','mp4','-o',str(output_template),
            *YTDLP_RUNTIME_ARGS,*attempt['extra'],row.source_url,
        ]
        print(f'Downloading {row.video_id}: {attempt["name"]}')
        result = subprocess.run(command, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, check=False)
        matches = sorted(p for p in VIDEO_ROOT.glob(f'{row.video_id}.*')
                         if p.suffix.lower() in VIDEO_SUFFIXES and p.stat().st_size > 1_000_000)
        if result.returncode == 0 and len(matches) == 1 and valid_video(matches[0]):
            return matches[0]
        for path in matches:
            path.unlink(missing_ok=True)
        tail = '\n'.join(result.stdout.splitlines()[-25:])
        failures.append(f'{attempt["name"]} (exit {result.returncode}):\n{tail}')
        print(tail)
    raise RuntimeError(f'All 360p attempts failed for {row.video_id}:\n\n' + '\n\n'.join(failures))

video_paths = {}
for row in sources.itertuples(index=False):
    path = download_360p(row)
    video_paths[row.video_id] = path
    print(row.video_id, probe_video(path))

## Extract denser RGB evidence around each draft locator

For each interval, ffmpeg samples 2 frames per second with 5 seconds of context on each side. Every contact sheet has 12 timestamped tiles. Timestamps are approximate to within a sample step; an independent reviewer must inspect visible motion and resolve uncertain edges before labels are locked.

In [ ]:
SAMPLE_FPS = 2
PAD_S = 5.0
TILE_W, TILE_H = 320, 180

def tile(frame, time_s, interval_id):
    h,w = frame.shape[:2]
    scale = min(TILE_W/w, TILE_H/h)
    resized = cv2.resize(frame, (max(1,round(w*scale)),max(1,round(h*scale))))
    canvas = np.zeros((TILE_H,TILE_W,3),dtype=np.uint8)
    x=(TILE_W-resized.shape[1])//2; y=(TILE_H-resized.shape[0])//2
    canvas[y:y+resized.shape[0],x:x+resized.shape[1]]=resized
    label=f'{time_s/60:.2f} min | {time_s:.1f}s'
    cv2.rectangle(canvas,(0,0),(TILE_W,25),(0,0,0),-1)
    cv2.putText(canvas,label,(5,18),cv2.FONT_HERSHEY_SIMPLEX,0.52,(0,255,255),1,cv2.LINE_AA)
    return canvas

index_rows=[]
for row in tqdm(intervals.itertuples(index=False), total=len(intervals), desc='Candidate regions'):
    video_path=video_paths[row.video_id]
    duration=probe_video(video_path)['duration_s']
    start=max(0.0,float(row.start_s)-PAD_S)
    end=min(duration,float(row.end_s)+PAD_S)
    interval_root=CONTACT_ROOT / row.interval_id
    interval_root.mkdir(parents=True,exist_ok=True)
    frame_root=FRAME_ROOT / row.interval_id
    if frame_root.exists():
        shutil.rmtree(frame_root)
    frame_root.mkdir(parents=True)
    command=[
        'ffmpeg','-hide_banner','-loglevel','error','-y',
        '-ss',str(start),'-i',str(video_path),'-t',str(end-start),
        '-vf',f'fps={SAMPLE_FPS},scale={TILE_W}:{TILE_H}:force_original_aspect_ratio=decrease,'
              f'pad={TILE_W}:{TILE_H}:(ow-iw)/2:(oh-ih)/2:black',
        '-q:v','3',str(frame_root/'%06d.jpg'),
    ]
    result=subprocess.run(command,text=True,stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT,check=False)
    frame_paths=sorted(frame_root.glob('*.jpg'))
    if result.returncode or len(frame_paths)<2:
        raise RuntimeError(f'ffmpeg failed for {row.interval_id}:\n{result.stdout[-2000:]}')
    for page_num in range((len(frame_paths)+11)//12):
        subset=frame_paths[page_num*12:(page_num+1)*12]
        tiles=[]
        for index,p in enumerate(subset,start=page_num*12):
            frame=cv2.imread(str(p))
            if frame is None: raise RuntimeError(f'Cannot read {p}')
            tiles.append(tile(frame,start+index/SAMPLE_FPS,row.interval_id))
        times=[start+(page_num*12+j)/SAMPLE_FPS for j in range(len(subset))]
        while len(tiles)<12: tiles.append(np.zeros_like(tiles[0]))
        page=np.concatenate([np.concatenate(tiles[r*4:(r+1)*4],axis=1)
                             for r in range(3)],axis=0)
        output=interval_root/f'{page_num:03d}.jpg'
        assert cv2.imwrite(str(output),page,[cv2.IMWRITE_JPEG_QUALITY,88])
        index_rows.append({
            'interval_id':row.interval_id,'video_id':row.video_id,
            'candidate_class':row.canonical_class,'page_index':page_num,
            'sample_fps':SAMPLE_FPS,'first_approx_s':times[0],
            'last_approx_s':times[-1], 'path':str(output.relative_to(OUTPUT_ROOT)),
        })
    shutil.rmtree(frame_root)
index=pd.DataFrame(index_rows)
assert set(index.interval_id)==set(intervals.interval_id)
index.to_csv(OUTPUT_ROOT/'contact_sheet_index.csv',index=False)
print('Contact sheet pages:',len(index))
display(index.groupby('video_id').agg(regions=('interval_id','nunique'),pages=('page_index','count')))

## Package and download one evidence ZIP

The archive contains draft locators, contact sheets, a page index, run metadata, and checksums. It contains no raw video, landmark rerun, training examples, or model predictions. Send the ZIP back for independent motion review and label audit.

In [ ]:
for name,path in input_paths.items():
    shutil.copy2(path,OUTPUT_ROOT/name)
metadata={
    'notebook':'29_colab_dense_creator_rgb_review.ipynb',
    'runtime':'CPU','pinned_data_commit':PINNED_DATA_COMMIT,
    'candidate_regions':len(intervals),'source_count':len(sources),
    'sample_fps':SAMPLE_FPS,'context_pad_s':PAD_S,
    'label_status':'draft_needs_dense_visual_review_and_independent_second_review',
    'training_allowed':False,'calibration_allowed':False,
    'threshold_selection_allowed':False,'model_predictions_included':False,
    'raw_video_included':False,
    'timestamp_note':'tile times approximate to one sample step after ffmpeg seek',
}
(OUTPUT_ROOT/'run_metadata.json').write_text(json.dumps(metadata,indent=2)+'\n')
assert not list(OUTPUT_ROOT.rglob('*.mp4'))
assert not list(OUTPUT_ROOT.rglob('*.webm'))
checksum_lines=[]
for path in sorted(p for p in OUTPUT_ROOT.rglob('*') if p.is_file() and p.name!='SHA256SUMS.txt'):
    checksum_lines.append(f'{sha256(path)}  {path.relative_to(OUTPUT_ROOT)}')
(OUTPUT_ROOT/'SHA256SUMS.txt').write_text('\n'.join(checksum_lines)+'\n')
archive=Path(shutil.make_archive(str(OUTPUT_ROOT),'zip',OUTPUT_ROOT))
print('Created:',archive,'size MB:',round(archive.stat().st_size/1e6,1))
print('SHA-256:',sha256(archive))
files.download(str(archive))